In [ ]:
import pandas as pd
import pickle
import arviz as az
from plotly import graph_objects as go
from datetime import datetime
import pycountry
from os import listdir as ls
from matplotlib_inline.backend_inline import set_matplotlib_formats
from matplotlib import pyplot as plt
import seaborn as sns

from emu_renewal.inputs import ANALYSIS_TYPES, DATA_PATH, BASE_PATH, OUTPUTS_PATH
from emu_renewal.outputs import load_targets
from emu_renewal.plotting import plot_spaghetti_calib_comparison, plot_post_prior_comparison, plot_imm_props

In [ ]:
set_matplotlib_formats("svg")

run_name = "42780561"
job_path = OUTPUTS_PATH / run_name
countries = ls(job_path)

In [ ]:
gmob_columns = [
    'retail_and_recreation',
    'grocery_and_pharmacy',
    'parks',
    'transit_stations',
    'workplaces',
    'residential'
]
amob3_columns = [
    "driving",
    "transit",
    "walking",
]
amob2_columns = [
    "driving",
    "walking",
]

In [ ]:
def plot_mob_weights_by_country(job_path, mob_type):
    fig, axes = plt.subplots(4, 4, figsize=[10, 10])
    flat_axes = axes.ravel()
    for c, country in enumerate(countries):
        c_path = job_path / country
        country_name = pycountry.countries.lookup(country).name
        idata = az.from_netcdf(c_path / f"weighted_{mob_type}_1exp/idata_filtered.nc")
        mob_weights = idata.posterior["mob_weights"].to_dataframe().unstack("mob_weights_dim_0")
        if mob_type == "google":
            mob_columns = gmob_columns
        elif mob_type == "apple" and mob_weights.shape[1] == 2:
            mob_columns = amob2_columns
        elif mob_type == "apple" and mob_weights.shape[1] == 3:
            mob_columns = amob3_columns
        else:
            raise ValueError("wrong mobility type request or number of mobility weights")
        mob_weights.columns = mob_columns
        c_ax = flat_axes[c]
        sns.kdeplot(mob_weights, fill=True, alpha=0.1, linewidth=1.5, ax=c_ax)
        c_ax.set_yticks([])
        c_ax.set_ylabel("")
        c_ax.set_title(country_name)
        if country != "FIN":
            flat_axes[c].get_legend().remove()
    fig.tight_layout()

In [ ]:
plot_mob_weights_by_country(job_path, "apple")

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
from itertools import product

def mw_plot(country, w=True, chain=0, ax=None,legend=True):
    c_path = job_path / country
    a_path = c_path / "weighted_google_1exp"        
    idata = az.from_netcdf(a_path / "idata_filtered.nc")
    
    if ax is None:
        fig = plt.figure();
        ax = fig.gca()
    if w:
        #plt.figure();
        df = idata.posterior["mob_weights"].sel(chain=chain).to_pandas()
        df.columns = gmob_columns
        sns.kdeplot(df,ax=ax,legend=legend)
        ax.set_title(f"{country}: mobility weights")
        return ax
    else:
        df = idata.posterior["mob_exp"].sel(chain=chain).to_pandas()
        df.columns = gmob_columns
        ax = sns.kdeplot(df,ax=ax,legend=legend)
        ax.set_title(f"{country}: mobility exponents")
        return ax

In [ ]:
#Plot weights and exponents for single country
country = "ITA"

mw_plot(country, True)
mw_plot(country, False)

In [ ]:
# Plot all weights in grid

fig, axes = plt.subplots(4,4,figsize=(16,16))
for cidx, aidx in enumerate(list(product(range(4),range(4)))):
    if cidx==0:
        legend = True
    else:
        legend = False
    try:
        # True/False switches between weights/exponents
        mw_plot(countries[cidx],True,ax=axes[aidx],legend=legend,chain=2)
    except:
        pass

In [ ]:
# Plot all exponents in grid

fig, axes = plt.subplots(4,4,figsize=(16,16))
for cidx, aidx in enumerate(list(product(range(4),range(4)))):
    if cidx==0:
        legend = True
    else:
        legend = False
    try:
        # True/False switches between weights/exponents
        mw_plot(countries[cidx],False,ax=axes[aidx],legend=legend,chain=3)
    except:
        pass